# Fair LiH Subspace Benchmark

This notebook fairly compares **Krylov**, **UCCSD**, and **fixed-architecture HEA** as subspace bases for **LiH**.

Fairness rules used here:
- same Hamiltonian
- same HF reference state
- same basis size `K`
- same projected-subspace objective
- same optimizer family and budget
- same conditioning rule for the overlap matrix `S`

This `LiH` notebook uses conservative default optimizer settings so the benchmark stays runnable while preserving the same fair subspace protocol as the `H2` benchmark.

The default setup uses `K_max = 3`, `maxiter = 20`, `maxfev = 120`, and `n_restarts = 2`.

This version uses a looser overlap cutoff `s_eval_cutoff = 1e-10` so the generalized eigensolver can keep more nearly dependent states than before.

If a basis family still saturates or fails to add another valid state at larger `K`, the notebook records that stop and continues with the other families.


In [ ]:
import os
import tempfile
import warnings
from pprint import pprint

os.environ.setdefault("MPLCONFIGDIR", os.path.join(tempfile.gettempdir(), "mpl"))
warnings.filterwarnings("ignore", message=r".*NumPy < 2\\.0\\.0.*")

import matplotlib.pyplot as plt
import numpy as np
import pennylane as qml
from scipy.optimize import minimize

DEFAULT_BOND_LENGTHS = {
    "H2": 0.74,
    "H4": 0.74,
    "LIH": 1.5474,
}

FAMILY_ORDER = ("krylov", "uccsd", "hea")
FAMILY_LABELS = {
    "krylov": "Krylov",
    "uccsd": "UCCSD",
    "hea": "Fixed HEA",
}

ROTATION_GATES = {"RX": qml.RX, "RY": qml.RY, "RZ": qml.RZ}
ENTANGLERS = {"CNOT": qml.CNOT, "CZ": qml.CZ}


def build_hamiltonian(mol_name: str, bond_length: float | None = None, basis: str = "sto-3g", unit: str = "angstrom"):
    name = mol_name.strip().upper()

    if name == "H2":
        bond = DEFAULT_BOND_LENGTHS[name] if bond_length is None else bond_length
        symbols = ["H", "H"]
        geometry = np.array([[0.0, 0.0, 0.0], [0.0, 0.0, bond]], dtype=float)
        electrons = 2
    elif name == "H4":
        bond = DEFAULT_BOND_LENGTHS[name] if bond_length is None else bond_length
        symbols = ["H", "H", "H", "H"]
        geometry = np.array(
            [[0.0, 0.0, 0.0], [0.0, 0.0, bond], [0.0, 0.0, 2.0 * bond], [0.0, 0.0, 3.0 * bond]],
            dtype=float,
        )
        electrons = 4
    elif name == "LIH":
        bond = DEFAULT_BOND_LENGTHS[name] if bond_length is None else bond_length
        symbols = ["Li", "H"]
        geometry = np.array([[0.0, 0.0, 0.0], [0.0, 0.0, bond]], dtype=float)
        electrons = 4
    else:
        raise ValueError("Supported molecules: H2, H4, LiH")

    H_op, n_qubits = qml.qchem.molecular_hamiltonian(
        symbols,
        geometry,
        charge=0,
        mult=1,
        basis=basis,
        unit=unit,
    )
    hf = np.array(qml.qchem.hf_state(electrons, n_qubits), dtype=int)
    return H_op, n_qubits, electrons, hf


def hf_statevector(hf_bitstring: np.ndarray) -> np.ndarray:
    n_qubits = len(hf_bitstring)
    index = int("".join(map(str, hf_bitstring.tolist())), 2)
    state = np.zeros(1 << n_qubits, dtype=complex)
    state[index] = 1.0
    return state


def normalize_statevector(state: np.ndarray) -> np.ndarray:
    state = np.asarray(state, dtype=complex)
    if not np.all(np.isfinite(state)):
        raise ValueError("Encountered a non-finite statevector.")
    norm = np.linalg.norm(state)
    if not np.isfinite(norm) or np.isclose(norm, 0.0):
        raise ValueError("Encountered an invalid or zero-norm statevector.")
    return state / norm


def convert_excitations_for_uccsd(singles, doubles):
    s_wires = [list(pair) for pair in singles]
    d_wires = []
    for i, j, a, b in doubles:
        d_wires.append([[int(i), int(j)], [int(a), int(b)]])
    return s_wires, d_wires


def normalize_rotation_gates(rotation_gates):
    normalized = [str(g).upper() for g in rotation_gates]
    if not normalized:
        raise ValueError("rotation_gates must contain at least one entry.")
    for gate in normalized:
        if gate not in ROTATION_GATES:
            raise ValueError(f"Unsupported rotation gate '{gate}'. Allowed: {tuple(ROTATION_GATES)}")
    return normalized


def validate_entangler(entangler: str, entangler_pattern: str) -> tuple[str, str]:
    entangler_name = str(entangler).upper()
    pattern_name = str(entangler_pattern).lower()
    if entangler_name not in ENTANGLERS:
        raise ValueError(f"Unsupported entangler '{entangler_name}'. Allowed: {tuple(ENTANGLERS)}")
    if pattern_name != "chain":
        raise ValueError("Only the 'chain' entangler pattern is supported in this benchmark notebook.")
    return entangler_name, pattern_name


def hea_param_count(n_qubits: int, n_layers: int, rotation_gates) -> int:
    return int(n_qubits) * int(n_layers) * len(rotation_gates)


def apply_entangler_layer(wires, entangler_name: str, entangler_pattern: str):
    if entangler_pattern == "chain":
        for i in range(len(wires) - 1):
            ENTANGLERS[entangler_name](wires=[wires[i], wires[i + 1]])


def apply_fixed_hea(theta, wires, n_layers: int, rotation_gates, entangler_name: str, entangler_pattern: str):
    cursor = 0
    for _ in range(n_layers):
        for gate_name in rotation_gates:
            gate_cls = ROTATION_GATES[gate_name]
            for wire in wires:
                gate_cls(theta[cursor], wires=wire)
                cursor += 1
        apply_entangler_layer(wires, entangler_name, entangler_pattern)


def count_pauli_terms(H_op) -> int:
    return len(qml.pauli.pauli_sentence(H_op))


def electron_sector_indices(n_qubits: int, electrons: int) -> list[int]:
    return [index for index in range(1 << n_qubits) if index.bit_count() == electrons]


def compute_fci_ground_energy(H_sparse, n_qubits: int, electrons: int) -> float:
    indices = electron_sector_indices(n_qubits, electrons)
    sector_matrix = H_sparse[indices, :][:, indices].toarray()
    sector_matrix = 0.5 * (sector_matrix + sector_matrix.conj().T)
    eigvals = np.linalg.eigvalsh(sector_matrix)
    return float(np.real(eigvals[0]))


def solve_projected_subspace(states, H_sparse, s_eval_cutoff: float):
    base_penalty = 1.0e6
    state_matrix = np.asarray(states, dtype=complex)
    if not np.all(np.isfinite(state_matrix)):
        return {
            "valid": False,
            "energy": base_penalty,
            "penalty": base_penalty,
            "overlap": None,
            "H_sub": None,
            "s_evals": None,
            "min_retained_s_eval": 0.0,
        }

    overlap = state_matrix.conj() @ state_matrix.T
    overlap = 0.5 * (overlap + overlap.conj().T)
    if not np.all(np.isfinite(overlap)):
        return {
            "valid": False,
            "energy": base_penalty,
            "penalty": base_penalty,
            "overlap": overlap,
            "H_sub": None,
            "s_evals": None,
            "min_retained_s_eval": 0.0,
        }

    s_evals, s_vecs = np.linalg.eigh(overlap)
    s_evals = np.real_if_close(s_evals).real
    if not np.all(np.isfinite(s_evals)):
        return {
            "valid": False,
            "energy": base_penalty,
            "penalty": base_penalty,
            "overlap": overlap,
            "H_sub": None,
            "s_evals": s_evals,
            "min_retained_s_eval": 0.0,
        }

    keep = s_evals > float(s_eval_cutoff)
    min_eval = float(np.min(s_evals))

    if keep.sum() != len(s_evals):
        penalty = 1.0e6 + 1.0e3 * float(np.sum(np.maximum(float(s_eval_cutoff) - s_evals, 0.0)))
        return {
            "valid": False,
            "energy": penalty,
            "penalty": penalty,
            "overlap": overlap,
            "H_sub": None,
            "s_evals": s_evals,
            "min_retained_s_eval": min_eval,
        }

    h_states = np.vstack([H_sparse @ psi for psi in state_matrix])
    h_sub = state_matrix.conj() @ h_states.T
    h_sub = 0.5 * (h_sub + h_sub.conj().T)
    if not np.all(np.isfinite(h_sub)):
        return {
            "valid": False,
            "energy": base_penalty,
            "penalty": base_penalty,
            "overlap": overlap,
            "H_sub": h_sub,
            "s_evals": s_evals,
            "min_retained_s_eval": min_eval,
        }

    transform = s_vecs[:, keep] / np.sqrt(s_evals[keep])[None, :]
    h_ortho = transform.conj().T @ h_sub @ transform
    h_ortho = 0.5 * (h_ortho + h_ortho.conj().T)
    if not np.all(np.isfinite(h_ortho)):
        return {
            "valid": False,
            "energy": base_penalty,
            "penalty": base_penalty,
            "overlap": overlap,
            "H_sub": h_sub,
            "s_evals": s_evals,
            "min_retained_s_eval": min_eval,
        }

    eigvals = np.linalg.eigvalsh(h_ortho)
    if eigvals.size == 0 or not np.all(np.isfinite(eigvals)):
        return {
            "valid": False,
            "energy": base_penalty,
            "penalty": base_penalty,
            "overlap": overlap,
            "H_sub": h_sub,
            "s_evals": s_evals,
            "min_retained_s_eval": min_eval,
        }

    energy = float(np.real(eigvals[0]))

    return {
        "valid": True,
        "energy": energy,
        "penalty": energy,
        "overlap": overlap,
        "H_sub": h_sub,
        "s_evals": s_evals,
        "min_retained_s_eval": float(np.min(s_evals[keep])),
    }


def build_system_context(
    mol_name: str,
    bond_length: float | None,
    basis: str,
    unit: str,
    krylov_trotter_steps: int,
    krylov_trotter_order: int,
    hea_n_layers: int,
    hea_rotation_gates,
    hea_entangler: str,
    hea_entangler_pattern: str,
    device_name: str,
):
    H_op, n_qubits, electrons, hf = build_hamiltonian(
        mol_name=mol_name,
        bond_length=bond_length,
        basis=basis,
        unit=unit,
    )
    wires = list(range(n_qubits))
    H_sparse = qml.pauli.pauli_sentence(H_op).to_mat(wire_order=wires, format="csr")
    hf_vector = hf_statevector(hf)

    singles, doubles = qml.qchem.excitations(electrons, n_qubits)
    s_wires, d_wires = convert_excitations_for_uccsd(singles, doubles)
    hea_rotation_gates = normalize_rotation_gates(hea_rotation_gates)
    hea_entangler, hea_entangler_pattern = validate_entangler(hea_entangler, hea_entangler_pattern)

    krylov_dev = qml.device(device_name, wires=wires)
    uccsd_dev = qml.device(device_name, wires=wires)
    hea_dev = qml.device(device_name, wires=wires)

    @qml.qnode(krylov_dev)
    def krylov_qnode(time_value):
        qml.BasisState(hf, wires=wires)
        qml.TrotterProduct(H_op, time=float(time_value), order=int(krylov_trotter_order), n=int(krylov_trotter_steps))
        return qml.state()

    hf_int = np.array(hf, dtype=int)

    @qml.qnode(uccsd_dev)
    def uccsd_qnode(theta):
        qml.UCCSD(
            np.asarray(theta, dtype=float),
            wires=wires,
            s_wires=s_wires,
            d_wires=d_wires,
            init_state=hf_int,
        )
        return qml.state()

    @qml.qnode(hea_dev)
    def hea_qnode(theta):
        qml.BasisState(hf_int, wires=wires)
        apply_fixed_hea(
            np.asarray(theta, dtype=float),
            wires=wires,
            n_layers=int(hea_n_layers),
            rotation_gates=hea_rotation_gates,
            entangler_name=hea_entangler,
            entangler_pattern=hea_entangler_pattern,
        )
        return qml.state()

    return {
        "mol_name": mol_name,
        "bond_length": DEFAULT_BOND_LENGTHS[mol_name.strip().upper()] if bond_length is None else float(bond_length),
        "basis": basis,
        "unit": unit,
        "H_op": H_op,
        "H_sparse": H_sparse,
        "n_terms": count_pauli_terms(H_op),
        "n_qubits": n_qubits,
        "electrons": electrons,
        "hf": hf_int,
        "hf_vector": hf_vector,
        "wires": wires,
        "krylov_qnode": krylov_qnode,
        "uccsd_qnode": uccsd_qnode,
        "hea_qnode": hea_qnode,
        "krylov_trotter_steps": int(krylov_trotter_steps),
        "krylov_trotter_order": int(krylov_trotter_order),
        "singles": [list(pair) for pair in singles],
        "doubles": [list(quad) for quad in doubles],
        "s_wires": [list(pair) for pair in s_wires],
        "d_wires": [[[int(a) for a in block] for block in pair] for pair in d_wires],
        "n_uccsd_params": len(s_wires) + len(d_wires),
        "hea_n_layers": int(hea_n_layers),
        "hea_rotation_gates": list(hea_rotation_gates),
        "hea_entangler": hea_entangler,
        "hea_entangler_pattern": hea_entangler_pattern,
        "n_hea_params": hea_param_count(n_qubits, hea_n_layers, hea_rotation_gates),
    }


def make_krylov_state(time_value, context):
    return normalize_statevector(context["krylov_qnode"](float(time_value)))


def make_uccsd_state(theta, context):
    return normalize_statevector(context["uccsd_qnode"](np.asarray(theta, dtype=float)))


def make_fixed_hea_state(theta, context):
    return normalize_statevector(context["hea_qnode"](np.asarray(theta, dtype=float)))


def family_param_dim(family: str, context) -> int:
    if family == "krylov":
        return 1
    if family == "uccsd":
        return int(context["n_uccsd_params"])
    if family == "hea":
        return int(context["n_hea_params"])
    raise ValueError(f"Unknown family '{family}'")


def family_bounds(family: str, context, t_max: float, theta_max: float):
    dim = family_param_dim(family, context)
    if family == "krylov":
        return [(0.0, float(t_max))] * dim
    return [(-float(theta_max), float(theta_max))] * dim


def normalize_candidate_params(family: str, params, context, t_max: float, theta_max: float):
    params = np.asarray(params, dtype=float).reshape(-1)
    if family == "krylov":
        return np.clip(params[:1], 0.0, float(t_max))
    dim = family_param_dim(family, context)
    return np.clip(params[:dim], -float(theta_max), float(theta_max))


def state_from_params(family: str, params, context):
    if family == "krylov":
        return make_krylov_state(params[0], context)
    if family == "uccsd":
        return make_uccsd_state(params, context)
    if family == "hea":
        return make_fixed_hea_state(params, context)
    raise ValueError(f"Unknown family '{family}'")


def spec_from_params(family: str, params):
    if family == "krylov":
        return {"time": float(params[0])}
    return {"theta": np.asarray(params, dtype=float).tolist()}


def total_free_params(family: str, n_optimized_states: int, context) -> int:
    if family == "krylov":
        return int(n_optimized_states)
    return int(n_optimized_states) * family_param_dim(family, context)


def initial_guesses_for_family(family: str, step_index: int, n_restarts: int, context, t_max: float, theta_max: float, random_seed: int):
    if int(n_restarts) <= 0:
        raise ValueError("n_restarts must be positive.")

    if family == "krylov":
        if int(n_restarts) == 1:
            return [np.array([0.5 * float(t_max)], dtype=float)]
        grid = np.linspace(0.0, float(t_max), int(n_restarts) + 2)[1:-1]
        return [np.array([value], dtype=float) for value in grid]

    dim = family_param_dim(family, context)
    family_offset = {"uccsd": 1000, "hea": 2000}[family]
    rng = np.random.default_rng(int(random_seed) + family_offset + 97 * int(step_index))
    guesses = [np.zeros(dim, dtype=float)]
    for _ in range(int(n_restarts) - 1):
        guess = rng.normal(loc=0.0, scale=0.15, size=dim)
        guesses.append(np.clip(guess, -float(theta_max), float(theta_max)).astype(float))
    return guesses


def evaluate_candidate_subspace(family: str, params, existing_states, context, t_max: float, theta_max: float, s_eval_cutoff: float):
    normalized = normalize_candidate_params(family, params, context, t_max=t_max, theta_max=theta_max)
    spec = spec_from_params(family, normalized)
    try:
        candidate_state = state_from_params(family, normalized, context)
        subspace = solve_projected_subspace(existing_states + [candidate_state], context["H_sparse"], s_eval_cutoff=float(s_eval_cutoff))
    except Exception:
        penalty = 1.0e6
        candidate_state = None
        subspace = {
            "valid": False,
            "energy": penalty,
            "penalty": penalty,
            "overlap": None,
            "H_sub": None,
            "s_evals": None,
            "min_retained_s_eval": 0.0,
        }
    return {
        "params": normalized,
        "state": candidate_state,
        "spec": spec,
        "subspace": subspace,
    }


def optimize_next_basis_state(
    family: str,
    step_index: int,
    existing_states,
    context,
    optimizer_method: str,
    maxiter: int,
    maxfev: int,
    n_restarts: int,
    random_seed: int,
    t_max: float,
    theta_max: float,
    s_eval_cutoff: float,
):
    bounds = family_bounds(family, context, t_max=t_max, theta_max=theta_max)
    initial_guesses = initial_guesses_for_family(
        family=family,
        step_index=step_index,
        n_restarts=n_restarts,
        context=context,
        t_max=t_max,
        theta_max=theta_max,
        random_seed=random_seed,
    )

    restart_summaries = []
    best = None

    for restart_index, x0 in enumerate(initial_guesses):
        objective = lambda x: evaluate_candidate_subspace(
            family=family,
            params=x,
            existing_states=existing_states,
            context=context,
            t_max=t_max,
            theta_max=theta_max,
            s_eval_cutoff=s_eval_cutoff,
        )["subspace"]["penalty"]

        result = minimize(
            objective,
            x0=np.asarray(x0, dtype=float),
            method=str(optimizer_method),
            bounds=bounds,
            options={"maxiter": int(maxiter), "maxfev": int(maxfev), "disp": False},
        )

        candidate = evaluate_candidate_subspace(
            family=family,
            params=result.x,
            existing_states=existing_states,
            context=context,
            t_max=t_max,
            theta_max=theta_max,
            s_eval_cutoff=s_eval_cutoff,
        )

        restart_summary = {
            "restart_index": int(restart_index),
            "initial_guess": np.asarray(x0, dtype=float).tolist(),
            "final_params": candidate["params"].tolist(),
            "success": bool(result.success),
            "message": str(result.message),
            "nfev": int(getattr(result, "nfev", -1)),
            "nit": int(getattr(result, "nit", -1)),
            "energy": float(candidate["subspace"]["energy"]),
            "valid": bool(candidate["subspace"]["valid"]),
        }
        restart_summaries.append(restart_summary)

        if best is None or candidate["subspace"]["energy"] < best["candidate"]["subspace"]["energy"]:
            best = {
                "restart_index": int(restart_index),
                "candidate": candidate,
                "optimizer_result": result,
            }

    if best is None or not best["candidate"]["subspace"]["valid"]:
        raise RuntimeError(f"Unable to find a valid {family} basis state at step {step_index}. Try reducing K_max or changing the ansatz settings for this molecule.")

    return {
        "state": best["candidate"]["state"],
        "spec": best["candidate"]["spec"],
        "subspace": best["candidate"]["subspace"],
        "optimizer_meta": {
            "k": int(step_index + 1),
            "winner_restart": int(best["restart_index"]),
            "best_fun": float(best["candidate"]["subspace"]["energy"]),
            "best_spec": best["candidate"]["spec"],
            "restart_summaries": restart_summaries,
        },
    }


def greedy_family_benchmark(
    family: str,
    context,
    K_max: int,
    optimizer_method: str,
    maxiter: int,
    maxfev: int,
    n_restarts: int,
    random_seed: int,
    t_max: float,
    theta_max: float,
    s_eval_cutoff: float,
    E_fci: float,
):
    if int(K_max) <= 0:
        raise ValueError("K_max must be positive.")

    basis_states = [context["hf_vector"]]
    selected_specs = [{"kind": "hf"}]
    hf_subspace = solve_projected_subspace(basis_states, context["H_sparse"], s_eval_cutoff=float(s_eval_cutoff))

    projected_energies = [float(hf_subspace["energy"])]
    energy_errors = [float(hf_subspace["energy"] - E_fci)]
    total_params = [0]
    min_retained_s_eval = [float(hf_subspace["min_retained_s_eval"])]
    optimizer_meta = []
    subspace_sizes = [1]
    stopped_early = False
    stop_k = None
    stop_reason = None

    for step_index in range(1, int(K_max)):
        try:
            best_step = optimize_next_basis_state(
                family=family,
                step_index=step_index,
                existing_states=basis_states,
                context=context,
                optimizer_method=optimizer_method,
                maxiter=maxiter,
                maxfev=maxfev,
                n_restarts=n_restarts,
                random_seed=random_seed,
                t_max=t_max,
                theta_max=theta_max,
                s_eval_cutoff=s_eval_cutoff,
            )
        except RuntimeError as exc:
            stopped_early = True
            stop_k = int(step_index + 1)
            stop_reason = str(exc)
            break

        basis_states.append(best_step["state"])
        selected_specs.append(best_step["spec"])
        projected_energies.append(float(best_step["subspace"]["energy"]))
        energy_errors.append(float(best_step["subspace"]["energy"] - E_fci))
        total_params.append(int(total_free_params(family, len(selected_specs) - 1, context)))
        min_retained_s_eval.append(float(best_step["subspace"]["min_retained_s_eval"]))
        optimizer_meta.append(best_step["optimizer_meta"])
        subspace_sizes.append(len(selected_specs))

    family_config = {
        "trotter_steps": int(context["krylov_trotter_steps"]),
        "trotter_order": int(context["krylov_trotter_order"]),
        "n_uccsd_params": int(context["n_uccsd_params"]),
        "hea_n_layers": int(context["hea_n_layers"]),
        "hea_rotation_gates": list(context["hea_rotation_gates"]),
        "hea_entangler": context["hea_entangler"],
        "hea_entangler_pattern": context["hea_entangler_pattern"],
        "n_hea_params": int(context["n_hea_params"]),
    }

    return {
        "family": family,
        "label": FAMILY_LABELS[family],
        "selected_specs": selected_specs,
        "basis_states": basis_states,
        "projected_energies": projected_energies,
        "energy_errors": energy_errors,
        "total_free_params": total_params,
        "min_retained_s_eval": min_retained_s_eval,
        "optimizer_meta": optimizer_meta,
        "subspace_sizes": subspace_sizes,
        "requested_K_max": int(K_max),
        "max_computed_k": int(subspace_sizes[-1]),
        "stopped_early": bool(stopped_early),
        "stop_k": None if stop_k is None else int(stop_k),
        "stop_reason": stop_reason,
        "family_config": family_config,
    }


def build_comparison_rows(results):
    rows = []
    for family in FAMILY_ORDER:
        family_result = results[family]
        for idx, K_value in enumerate(family_result["subspace_sizes"]):
            rows.append(
                {
                    "family": family,
                    "label": family_result["label"],
                    "K": int(K_value),
                    "projected_energy": float(family_result["projected_energies"][idx]),
                    "energy_error": float(family_result["energy_errors"][idx]),
                    "total_free_params": int(family_result["total_free_params"][idx]),
                    "min_retained_s_eval": float(family_result["min_retained_s_eval"][idx]),
                }
            )
    return rows


In [ ]:
mol_name = "LiH"
bond_length = 1.5474
basis = "sto-3g"
unit = "angstrom"
K_max = 3
optimizer_method = "Powell"
maxiter = 20
maxfev = 120
n_restarts = 2
random_seed = 1234
t_max = 1.0
theta_max = 1.0
s_eval_cutoff = 1.0e-10
device_name = "lightning.qubit"
krylov_trotter_steps = 2
krylov_trotter_order = 1
hea_n_layers = 1
hea_rotation_gates = ["RY", "RZ"]
hea_entangler = "CNOT"
hea_entangler_pattern = "chain"


In [ ]:
context = build_system_context(
    mol_name=mol_name,
    bond_length=bond_length,
    basis=basis,
    unit=unit,
    krylov_trotter_steps=krylov_trotter_steps,
    krylov_trotter_order=krylov_trotter_order,
    hea_n_layers=hea_n_layers,
    hea_rotation_gates=hea_rotation_gates,
    hea_entangler=hea_entangler,
    hea_entangler_pattern=hea_entangler_pattern,
    device_name=device_name,
)

E_fci = compute_fci_ground_energy(context["H_sparse"], context["n_qubits"], context["electrons"])

results = {}
for family in FAMILY_ORDER:
    results[family] = greedy_family_benchmark(
        family=family,
        context=context,
        K_max=K_max,
        optimizer_method=optimizer_method,
        maxiter=maxiter,
        maxfev=maxfev,
        n_restarts=n_restarts,
        random_seed=random_seed,
        t_max=t_max,
        theta_max=theta_max,
        s_eval_cutoff=s_eval_cutoff,
        E_fci=E_fci,
    )

comparison_rows = build_comparison_rows(results)
best_specs_by_family = {family: results[family]["selected_specs"] for family in FAMILY_ORDER}


In [ ]:
print(f"Molecule: {mol_name}")
print(f"FCI reference energy: {E_fci:.12f} Ha")
print(f"n_qubits: {context['n_qubits']}  electrons: {context['electrons']}  n_terms: {context['n_terms']}")
print()
print("Comparison rows:")
for row in comparison_rows:
    print(
        f"{row['label']:>9s} | K={row['K']} | E_sub={row['projected_energy']:.12f} | "
        f"E_sub-E_FCI={row['energy_error']:+.12f} | params={row['total_free_params']:>2d} | "
        f"min_S_eval={row['min_retained_s_eval']:.3e}"
    )

print()
print("Family stop status:")
for family in FAMILY_ORDER:
    family_result = results[family]
    if family_result["stopped_early"]:
        print(
            f"{family_result['label']:>9s} stopped before K={family_result['stop_k']}: "
            f"{family_result['stop_reason']}"
        )
    else:
        print(
            f"{family_result['label']:>9s} reached K={family_result['max_computed_k']} "
            f"of requested K={family_result['requested_K_max']}"
        )

print()
print("Selected parameter specs by family:")
pprint(best_specs_by_family)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

for family in FAMILY_ORDER:
    family_result = results[family]
    ks = family_result['subspace_sizes']
    axes[0].plot(ks, family_result['projected_energies'], marker='o', label=family_result['label'])
    axes[1].plot(ks, family_result['energy_errors'], marker='o', label=family_result['label'])
    axes[2].plot(ks, family_result['total_free_params'], marker='o', label=family_result['label'])
    axes[3].plot(ks, family_result['min_retained_s_eval'], marker='o', label=family_result['label'])

axes[0].axhline(E_fci, color='black', linestyle='--', linewidth=1.0, label='FCI')
axes[0].set_title('Projected Energy vs K')
axes[0].set_xlabel('Basis size K')
axes[0].set_ylabel('Energy (Ha)')

axes[1].set_title('Projected Energy Error vs K')
axes[1].set_xlabel('Basis size K')
axes[1].set_ylabel(r'$E_{sub} - E_{FCI}$ (Ha)')

axes[2].set_title('Total Free Parameters vs K')
axes[2].set_xlabel('Basis size K')
axes[2].set_ylabel('Total free parameters')

axes[3].set_title('Smallest Retained S Eigenvalue vs K')
axes[3].set_xlabel('Basis size K')
axes[3].set_ylabel('min retained eig(S)')

for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.legend()

fig.tight_layout()
plt.show()

gap_fig, gap_ax = plt.subplots(figsize=(8, 5))
family_colors = {
    'krylov': 'tab:blue',
    'uccsd': 'tab:orange',
    'hea': 'tab:green',
}
family_linestyles = {
    'krylov': '-',
    'uccsd': ':',
    'hea': '-',
}

for family in FAMILY_ORDER:
    family_result = results[family]
    gap_ax.plot(
        family_result['subspace_sizes'],
        family_result['energy_errors'],
        marker='o',
        linewidth=2.0,
        color=family_colors[family],
        linestyle=family_linestyles[family],
        label=family_result['label'],
    )

gap_ax.set_title('LiH Energy Gap vs Basis Size K')
gap_ax.set_xlabel('Number of basis states, K')
gap_ax.set_ylabel(r'$E_{sub} - E_{FCI}$ (Ha)')
gap_ax.grid(True, alpha=0.3)
gap_ax.legend()
gap_fig.tight_layout()

gap_plot_png = os.path.join(os.getcwd(), 'Fair_Subspace_Benchmark_LiH_Gap_vs_K.png')
gap_plot_pdf = os.path.join(os.getcwd(), 'Fair_Subspace_Benchmark_LiH_Gap_vs_K.pdf')
gap_fig.savefig(gap_plot_png, dpi=200, bbox_inches='tight')
gap_fig.savefig(gap_plot_pdf, bbox_inches='tight')
print(f'Saved gap plot to: {gap_plot_png}')
print(f'Saved gap plot to: {gap_plot_pdf}')
plt.show()
plt.close(gap_fig)


In [ ]:
for family in FAMILY_ORDER:
    family_result = results[family]
    assert family_result['selected_specs'][0]['kind'] == 'hf'
    energies = family_result['projected_energies']
    assert all(energies[i + 1] <= energies[i] + 1.0e-8 for i in range(len(energies) - 1))
    assert family_result['max_computed_k'] == family_result['subspace_sizes'][-1]
    if family_result['stopped_early']:
        assert family_result['stop_k'] is not None
        assert family_result['max_computed_k'] < family_result['requested_K_max']
    else:
        assert family_result['max_computed_k'] == family_result['requested_K_max']

assert results['hea']['family_config']['hea_n_layers'] == hea_n_layers
assert results['hea']['family_config']['hea_rotation_gates'] == normalize_rotation_gates(hea_rotation_gates)
assert results['hea']['family_config']['hea_entangler'] == validate_entangler(hea_entangler, hea_entangler_pattern)[0]
assert results['hea']['family_config']['hea_entangler_pattern'] == validate_entangler(hea_entangler, hea_entangler_pattern)[1]

print('Sanity checks passed.')
